# 2021 Local Government Election Data Cleaning

This notebook builds the 2021 voting-district dataset used in the analysis. It combines the nine provincial result files published by the IEC, checks the data for missing, impossible and duplicate values, and saves one cleaned file.

**Why the files are combined in code:** an earlier version combined the provinces in Excel, which holds at most 1,048,576 rows. The 2021 data has 1,084,734 rows, so Excel silently cut off most of the Western Cape (the last province loaded). Combining the files with pandas keeps every row, and the completeness check at the end confirms all 213 municipalities are present.

In [1]:
import pandas as pd
from pathlib import Path

# Paths are relative to the project folder, so this notebook runs on any computer.
# It works whether Jupyter is opened in the notebooks/ folder or in the project root.
PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = PROJECT_DIR / "data" / "raw" / "2021_provincial"
CLEAN_DIR = PROJECT_DIR / "data" / "cleaned"

## Load and combine the nine provincial files

Each file is zipped to stay under GitHub's file-size limit; pandas reads the zip directly. We record which file each row came from in `Source.Name`.

In [2]:
parts = []
for file in sorted(RAW_DIR.glob("*.csv.zip")):
    part = pd.read_csv(file, encoding="utf-8-sig")   # utf-8-sig removes the hidden marker at the start of IEC files
    part.insert(0, "Source.Name", file.name.removesuffix(".zip"))
    parts.append(part)
    print(f"{file.name:<18} {len(part):>9,} rows  {part['Municipality'].nunique():>3} municipalities")

df = pd.concat(parts, ignore_index=True)
print(f"\nCombined: {len(df):,} rows and {df['Municipality'].nunique()} municipalities")
print(df.head())

EC_2021.csv.zip      156,083 rows   33 municipalities
FS_2021.csv.zip       61,123 rows   19 municipalities


GP.csv.zip           195,424 rows    9 municipalities


KN_2021.csv.zip      251,001 rows   44 municipalities


LP_2021.csv.zip      160,412 rows   22 municipalities
MP_2021.csv.zip       62,277 rows   17 municipalities
NC_2021.csv.zip       17,876 rows   26 municipalities


NW_2021.csv.zip       75,679 rows   18 municipalities
WC_2021.csv.zip      104,859 rows   25 municipalities



Combined: 1,084,734 rows and 213 municipalities
   Source.Name      Province        Municipality           Ward  \
0  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   
1  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   
2  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   
3  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   
4  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   

   VotingDistrict   VotingStationName  RegisteredVoters BallotType  \
0        10590151  PEFFERVILLE CLINIC              2724         PR   
1        10590151  PEFFERVILLE CLINIC              2724         PR   
2        10590151  PEFFERVILLE CLINIC              2724         PR   
3        10590151  PEFFERVILLE CLINIC              2724         PR   
4        10590151  PEFFERVILLE CLINIC              2724         PR   

   SpoiltVotes                                PartyName  TotalValidVotes  \
0            1                    ABANTU BATHO CONG

## Inspect the structure, missing values and value ranges

In [3]:
# Check the number of rows and columns
print("Rows and columns:", df.shape)

# Show all column names
print("\nColumn names:")
print(df.columns.tolist())

# Show the data types of each column
print("\nData types:")
print(df.dtypes)

Rows and columns: (1084734, 12)

Column names:
['Source.Name', 'Province', 'Municipality', 'Ward', 'VotingDistrict', 'VotingStationName', 'RegisteredVoters', 'BallotType', 'SpoiltVotes', 'PartyName', 'TotalValidVotes', 'DateGenerated']

Data types:
Source.Name            str
Province               str
Municipality           str
Ward                   str
VotingDistrict       int64
VotingStationName      str
RegisteredVoters     int64
BallotType             str
SpoiltVotes          int64
PartyName              str
TotalValidVotes      int64
DateGenerated          str
dtype: object


In [4]:
df.isnull().sum()

Source.Name            0
Province               0
Municipality           0
Ward                   0
VotingDistrict         0
VotingStationName    170
RegisteredVoters       0
BallotType             0
SpoiltVotes            0
PartyName              0
TotalValidVotes        0
DateGenerated          0
dtype: int64

In [5]:
#Inspecting the missing rows
df[df["Province"].isnull()]

,Source.Name,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated


In [6]:
df[df["VotingStationName"].isnull()]

,Source.Name,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated
1023622,WC_2021.csv,Western Cape,CPT - City of Cape Town,Ward 19100075,97091714,NaN,2468,PR,20,ABANTU BATHO CONGRESS,0,5/11/2022 11:32:31 PM
1023623,WC_2021.csv,Western Cape,CPT - City of Cape Town,Ward 19100075,97091714,NaN,2468,PR,20,AFRICA RESTORATION ALLIANCE,13,5/11/2022 11:32:31 PM
1023624,WC_2021.csv,Western Cape,CPT - City of Cape Town,Ward 19100075,97091714,NaN,2468,PR,20,AFRICAN CHRISTIAN DEMOCRATIC PARTY,94,5/11/2022 11:32:31 PM
1023625,WC_2021.csv,Western Cape,CPT - City of Cape Town,Ward 19100075,97091714,NaN,2468,PR,20,AFRICAN FREEDOM REVOLUTION,0,5/11/2022 11:32:31 PM
1023626,WC_2021.csv,Western Cape,CPT - City of Cape Town,Ward 19100075,97091714,NaN,2468,PR,20,AFRICAN INDEPENDENT CONGRESS,0,5/11/2022 11:32:31 PM
...,...,...,...,...,...,...,...,...,...,...,...,...
1046383,WC_2021.csv,Western Cape,CPT - City of Cape Town,Ward 19100116,97091422,NaN,2673,Ward,17,UNITED DEMOCRATIC MOVEMENT,2,5/11/2022 11:32:31 PM
1046384,WC_2021.csv,Western Cape,CPT - City of Cape Town,Ward 19100116,97091422,NaN,2673,Ward,17,UNITED INDEPENDENT MOVEMENT,5,5/11/2022 11:32:31 PM
1046385,WC_2021.csv,Western Cape,CPT - City of Cape Town,Ward 19100116,97091422,NaN,2673,Ward,17,UNITED PROGRESSIVE PARTY SOUTH AFRICA,0,5/11/2022 11:32:31 PM
1046386,WC_2021.csv,Western Cape,CPT - City of Cape Town,Ward 19100116,97091422,NaN,2673,Ward,17,UNITED SOUTH AFRICA,4,5/11/2022 11:32:31 PM


In [7]:
#Needed to understand what types of election records we are dealing with
df["BallotType"].value_counts(dropna=False)

BallotType
PR        468765
Ward      330745
DC 40%    285224
Name: count, dtype: int64

In [8]:
df["Province"].value_counts(dropna=False)

Province
KwaZulu-Natal    251001
Gauteng          195424
Limpopo          160412
Eastern Cape     156083
Western Cape     104859
North West        75679
Mpumalanga        62277
Free State        61123
Northern Cape     17876
Name: count, dtype: int64

In [9]:
df["RegisteredVoters"].describe()

count    1.084734e+06
mean     1.334306e+03
std      1.102484e+03
min      2.000000e+00
25%      5.000000e+02
50%      1.008000e+03
75%      1.908000e+03
max      8.237000e+03
Name: RegisteredVoters, dtype: float64

In [10]:
df["SpoiltVotes"].describe()

count    1.084734e+06
mean     1.058801e+01
std      1.662799e+01
min      0.000000e+00
25%      2.000000e+00
50%      6.000000e+00
75%      1.300000e+01
max      6.440000e+02
Name: SpoiltVotes, dtype: float64

In [11]:
df["TotalValidVotes"].describe()

count    1.084734e+06
mean     2.820782e+01
std      1.056575e+02
min      0.000000e+00
25%      0.000000e+00
50%      1.000000e+00
75%      6.000000e+00
max      3.567000e+03
Name: TotalValidVotes, dtype: float64

## Clean the records

Remove any completely blank rows, label missing voting-station names, and trim stray spaces from text fields.

In [12]:
# Remove the 8 completely blank election records
# These rows have no Province/election information
df = df[df["Province"].notna()].copy()

# Check the new dataset size after cleaning
print(df.shape)

(1084734, 12)


In [13]:
# Replace missing voting station names with a clear label
# to keep these rows because they contain valid election information
df["VotingStationName"] = df["VotingStationName"].fillna("Unknown")

# Check that no voting station names are missing anymore
print(df["VotingStationName"].isnull().sum())

0


In [14]:
# Remove accidental spaces at the beginning or end of text values
# This will prevent "Gauteng" and "Gauteng " from being treated as different values
text_columns = [
    "Source.Name",
    "Province",
    "Municipality",
    "Ward",
    "VotingStationName",
    "BallotType",
    "PartyName"
]

for column in text_columns:
    df[column] = df[column].str.strip()

# Check the first few rows to confirm the text fields still look correct
print(df[text_columns].head())

   Source.Name      Province        Municipality           Ward  \
0  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   
1  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   
2  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   
3  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   
4  EC_2021.csv  Eastern Cape  BUF - Buffalo City  Ward 29200001   

    VotingStationName BallotType                                PartyName  
0  PEFFERVILLE CLINIC         PR                    ABANTU BATHO CONGRESS  
1  PEFFERVILLE CLINIC         PR              AFRICA RESTORATION ALLIANCE  
2  PEFFERVILLE CLINIC         PR       AFRICAN CHRISTIAN DEMOCRATIC PARTY  
3  PEFFERVILLE CLINIC         PR             AFRICAN INDEPENDENT CONGRESS  
4  PEFFERVILLE CLINIC         PR  AFRICAN MULTICULTURAL ECONOMIC CONGRESS  


## Validate the counts

Check for missing or negative counts, duplicated records, the party list and how ballot types are distributed.

In [15]:
# Check for missing values in the numeric election fields
# These fields are important for calculating voter turnout and analysing
numeric_columns = [
    "VotingDistrict",
    "RegisteredVoters",
    "SpoiltVotes",
    "TotalValidVotes"
]

print(df[numeric_columns].isnull().sum())

VotingDistrict      0
RegisteredVoters    0
SpoiltVotes         0
TotalValidVotes     0
dtype: int64


In [16]:
# Check whether any vote counts are negative
# Vote counts cannot be negative so any negative values would need investigation
print("Negative RegisteredVoters:", (df["RegisteredVoters"] < 0).sum())
print("Negative SpoiltVotes:", (df["SpoiltVotes"] < 0).sum())
print("Negative TotalValidVotes:", (df["TotalValidVotes"] < 0).sum())

Negative RegisteredVoters: 0
Negative SpoiltVotes: 0
Negative TotalValidVotes: 0


In [17]:
# Identify records that have the same location, ballot type and party
# inspecting them for now nothing will be deleted
record_columns = [
    "Province",
    "Municipality",
    "Ward",
    "VotingDistrict",
    "BallotType",
    "PartyName"
]

duplicates = df[df.duplicated(subset=record_columns, keep=False)]

# Show how many potentially duplicated records we found
print("Potential duplicate records:", len(duplicates))


duplicates.head(20)

Potential duplicate records: 0


,Source.Name,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated


In [18]:
# Inspecting the party names 
# We are checking whether every value represents an actual party or result entry
print("Number of unique party names:", df["PartyName"].nunique())

# Display all party names in alphabetical order
print(df["PartyName"].dropna().sort_values().unique())

Number of unique party names: 324


<StringArray>
[           'ABAHLALI BASE MKHANYAKUDE MOVEMENT',
                                'ABAHLALY BAAHI',
                         'ABANTU BATHO CONGRESS',
                     'ABANTU INTEGRITY MOVEMENT',
                               'ABLE LEADERSHIP',
                       'ACADEMIC CONGRESS UNION',
                                      'ACTIONSA',
                     'ACTIVE CITIZENS COALITION',
                    'ACTIVE MOVEMENT FOR CHANGE',
              'ACTIVE NATION AGAINST CORRUPTION',
 ...
                'VOTER'S INDEPENDENT PARTY - SA',
                            'VRYHEIDSFRONT PLUS',
                        'WESTERN PROVINCE PARTY',
                              'WITZENBERG AKSIE',
    'WITZENBERG ONAFHANKLIKE DEMOKRATIESE PARTY',
                              'WITZENBERG PARTY',
                                  'XIMOKO PARTY',
                           'YOUNG PEOPLES PARTY',
 'YOUTH INDEPENDENCE PARTY AND YOUTH ASSOCIATES',
                           'ZUL

In [19]:
# Count election records by ballot type
# This helps us understand how PR, Ward and DC 40% results are represented 
# before we calculate turnout or political competition measures.

ballot_party_counts = (
    df.groupby("BallotType")["PartyName"]
      .nunique()
      .sort_values(ascending=False)
)

print(ballot_party_counts)

BallotType
Ward      320
PR        312
DC 40%    172
Name: PartyName, dtype: int64


In [20]:
# Check whether voting districts appear under more than one ballot type
# A voting district can legitimately have Ward, PR and DC 40% results,
# so decided inspect the structure before doing any aggregation.
district_ballot_counts = (
    df.groupby("VotingDistrict")["BallotType"]
      .nunique()
)
print(district_ballot_counts.value_counts().sort_index())

BallotType
2     4942
3    18205
Name: count, dtype: int64


In [21]:
# Find the one voting district that has only one ballot type
single_ballot_districts = district_ballot_counts[
    district_ballot_counts == 1
].index

df[df["VotingDistrict"].isin(single_ballot_districts)]

,Source.Name,Province,Municipality,Ward,VotingDistrict,VotingStationName,RegisteredVoters,BallotType,SpoiltVotes,PartyName,TotalValidVotes,DateGenerated


## Standardise types and run the final checks

In [22]:
cleaned_df = df.copy()

# Convert ID and count columns to integers
cleaned_df["VotingDistrict"] = cleaned_df["VotingDistrict"].astype("int64")
cleaned_df["RegisteredVoters"] = cleaned_df["RegisteredVoters"].astype("int64")
cleaned_df["SpoiltVotes"] = cleaned_df["SpoiltVotes"].astype("int64")
cleaned_df["TotalValidVotes"] = cleaned_df["TotalValidVotes"].astype("int64")

# Remove any accidental spaces from text fields
text_columns = [
    "Source.Name",
    "Province",
    "Municipality",
    "Ward",
    "VotingStationName",
    "BallotType",
    "PartyName"
]
for column in text_columns:
    cleaned_df[column] = cleaned_df[column].str.strip()

# Confirm the cleaned dataset
print("Dataset size:", cleaned_df.shape)
print("\nMissing values:")
print(cleaned_df.isnull().sum())

Dataset size: (1084734, 12)

Missing values:


Source.Name          0
Province             0
Municipality         0
Ward                 0
VotingDistrict       0
VotingStationName    0
RegisteredVoters     0
BallotType           0
SpoiltVotes          0
PartyName            0
TotalValidVotes      0
DateGenerated        0
dtype: int64


In [23]:
# Check for impossible vote counts
# A party should not have more valid votes than the registered voters
# at the same voting district.
invalid_votes = cleaned_df[
    cleaned_df["TotalValidVotes"] > cleaned_df["RegisteredVoters"]
]

print("Records where valid votes exceed registered voters:",
      len(invalid_votes))
print(invalid_votes.head(10))

Records where valid votes exceed registered voters: 57
        Source.Name      Province      Municipality           Ward  \
50305   EC_2021.csv  Eastern Cape  EC126 - Ngqushwa  Ward 21206012   
50318   EC_2021.csv  Eastern Cape  EC126 - Ngqushwa  Ward 21206012   
50323   EC_2021.csv  Eastern Cape  EC126 - Ngqushwa  Ward 21206012   
77199   EC_2021.csv  Eastern Cape  EC141 - Elundini  Ward 21401007   
77208   EC_2021.csv  Eastern Cape  EC141 - Elundini  Ward 21401007   
77214   EC_2021.csv  Eastern Cape  EC141 - Elundini  Ward 21401007   
79173   EC_2021.csv  Eastern Cape  EC141 - Elundini  Ward 21401017   
79182   EC_2021.csv  Eastern Cape  EC141 - Elundini  Ward 21401017   
79188   EC_2021.csv  Eastern Cape  EC141 - Elundini  Ward 21401017   
105445  EC_2021.csv  Eastern Cape  EC156 - Mhlontlo  Ward 21506025   

        VotingDistrict                    VotingStationName  RegisteredVoters  \
50305         10920035          BENTON LOWER PRIMARY SCHOOL               452   
50318       

In [24]:
# Standardise the numeric columns
# These columns contain IDs or counts, so decimal places are not needed.
numeric_columns = [
    "VotingDistrict",
    "RegisteredVoters",
    "SpoiltVotes",
    "TotalValidVotes"
]
for column in numeric_columns:
    cleaned_df[column] = cleaned_df[column].astype("int64")
print(cleaned_df[numeric_columns].dtypes)

VotingDistrict      int64
RegisteredVoters    int64
SpoiltVotes         int64
TotalValidVotes     int64
dtype: object


In [25]:
# Checking the dates represented in the dataset
# to confirm that the dataset is consistently from the
# 2021 local government election data.
print("Earliest date:", cleaned_df["DateGenerated"].min())
print("Latest date:", cleaned_df["DateGenerated"].max())
print("Number of unique dates:", cleaned_df["DateGenerated"].nunique())

Earliest date: 1/28/2022 12:10:24 PM
Latest date: 5/11/2022 11:32:31 PM


Number of unique dates: 8


In [26]:
# Final quality check before saving the cleaned dataset
# checked the main conditions cleaned for.
print("Rows:", len(cleaned_df))
print("Columns:", len(cleaned_df.columns))
print("\nMissing values:", cleaned_df.isnull().sum().sum())
print("Exact duplicate rows:", cleaned_df.duplicated().sum())
print(
    "Negative RegisteredVoters:",
    (cleaned_df["RegisteredVoters"] < 0).sum()
)
print(
    "Negative SpoiltVotes:",
    (cleaned_df["SpoiltVotes"] < 0).sum()
)
print(
    "Negative TotalValidVotes:",
    (cleaned_df["TotalValidVotes"] < 0).sum()
)

Rows: 1084734
Columns: 12



Missing values: 0


Exact duplicate rows: 0
Negative RegisteredVoters: 0
Negative SpoiltVotes: 0
Negative TotalValidVotes: 0


## Completeness check

Every metropolitan and local municipality must be present: 213 nationally, including all 25 in the Western Cape.

In [27]:
per_province = cleaned_df.groupby("Province")["Municipality"].nunique()
print(per_province.to_string())
print(f"\nTotal municipalities: {cleaned_df['Municipality'].nunique()}")
assert cleaned_df["Municipality"].nunique() == 213, "Incomplete 2021 data: check the provincial files"
assert per_province["Western Cape"] == 25

Province
Eastern Cape     33
Free State       19
Gauteng           9
KwaZulu-Natal    44
Limpopo          22
Mpumalanga       17
North West       18
Northern Cape    26
Western Cape     25



Total municipalities: 213


## Save the cleaned dataset

The cleaned file is about 150 MB, over GitHub's 100 MB limit, so it is saved as a zip. The EDA notebook reads the zip directly.

In [28]:
cleaned_df.to_csv(
    CLEAN_DIR / "LGE2021_Cleaned_Level.csv.zip",
    index=False,
    compression={"method": "zip", "archive_name": "LGE2021_Cleaned_Level.csv"},
)
print("Saved: data/cleaned/LGE2021_Cleaned_Level.csv.zip")

Saved: data/cleaned/LGE2021_Cleaned_Level.csv.zip
